# LeetCode 4077 — Minimize Inversions in a Prefix Tree (Hard)

Bài toán LeetCode:

```python
class Solution:
    def minInversions(self, words: List[str], target: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4077 (Biweekly Contest 193 — câu 4, rating khoảng 2485).

Bài này kết hợp bốn chủ đề:

1. **Cây tiền tố (trie)** và **duyệt tiền thứ tự (pre-order)**.
2. **Nghịch thế (inversion)** của một dãy.
3. Quan sát then chốt: số nghịch thế **tách được** thành các phần **độc lập** theo từng nút của trie.
4. Tại mỗi nút có tối đa 10 con (chữ `a..j`) → chọn thứ tự con tốt nhất bằng **DP trên tập con (bitmask DP)**.

---

## Sau notebook này, bạn sẽ nắm được

1. Dựng trie, duyệt pre-order, đếm nghịch thế.
2. Brute Force thử mọi cách sắp con ở mọi nút.
3. Phân rã số nghịch thế thành: (a) trong từng cây con, (b) giữa từ ở nút và cây con bên dưới, (c) giữa các cây con anh em.
4. Đếm số cặp nghịch thế giữa hai danh sách đã sắp bằng `bisect`.
5. Bitmask DP tìm thứ tự tối ưu của `k <= 10` phần tử với chi phí theo cặp.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng `words` gồm `n` chuỗi **khác nhau** (chỉ dùng chữ `a` tới `j`) và `target` là một **hoán vị** của `0..n-1`.

**Cây tiền tố:** gốc là chuỗi rỗng; mỗi nút khác là một tiền tố khác rỗng của một từ nào đó; cha của một nút là tiền tố bỏ đi ký tự cuối. Nút ứng với `words[i]` được **đánh dấu** bằng chỉ số `i` (nút đánh dấu vẫn có thể có con).

Ta được **tự chọn thứ tự các con** ở mỗi nút (độc lập với nhau), nhưng không đổi quan hệ cha–con.

**Duyệt tiền thứ tự (pre-order)** từ gốc: tại mỗi nút, nếu nút được đánh dấu thì ghi chỉ số của nó, rồi lần lượt duyệt hết cây con của từng con theo thứ tự đã chọn. Ta được mảng `traversal` dài `n`.

Gọi `pos[i]` = vị trí của `i` trong `target`. Một **nghịch thế** là cặp vị trí `a < b` trong `traversal` mà `pos[traversal[a]] > pos[traversal[b]]`.

Trả về **số nghịch thế nhỏ nhất** trên mọi cách chọn thứ tự con.

---

## Ví dụ 1

```python
words = ["bad", "bag", "fig", "fed"], target = [3, 1, 0, 2]   ->   2
```

`pos = [2, 1, 3, 0]` (từ 0 = "bad" ở vị trí 2 của target, ...).

Thăm nhánh `"f"` trước `"b"`; trong `"f"` thăm `"fed"` trước `"fig"`; trong `"b"` thăm `"bag"` trước `"bad"` → `traversal = [3, 2, 1, 0]`, dãy `pos` tương ứng `[0, 3, 1, 2]` → 2 nghịch thế (`3 > 1`, `3 > 2`). Không cách nào tốt hơn.

## Ví dụ 2

```python
words = ["ab", "ac", "ba", "bc"], target = [0, 3, 1, 2]   ->   1
```

`traversal = [0, 1, 3, 2]`, dãy `pos` = `[0, 2, 1, 3]` → 1 nghịch thế.

---

## Ràng buộc

```text
1 <= n <= 10^4
1 <= words[i].length <= 20, chỉ gồm 'a'..'j', các từ khác nhau
target là hoán vị của 0..n-1
```

# 2. Kiến thức nền tảng

## 2.1. Trie (cây tiền tố)

Mỗi nút giữ một `dict` ký tự → nút con. Chèn một từ = đi theo từng ký tự, tạo nút nếu chưa có. Ta đánh số nút theo thứ tự tạo: **nút con luôn có số lớn hơn nút cha** — duyệt số giảm dần là xử lý con trước cha.

In [1]:
from typing import List


def build_trie(words: List[str]):
    children = [{}]          # children[u]: ký tự -> nút con
    mark = [-1]              # mark[u]: chỉ số từ kết thúc tại u, -1 nếu không có
    for i, w in enumerate(words):
        u = 0
        for c in w:
            if c not in children[u]:
                children[u][c] = len(children)
                children.append({})
                mark.append(-1)
            u = children[u][c]
        mark[u] = i
    return children, mark


children, mark = build_trie(["bad", "bag", "fig", "fed"])
for u in range(len(children)):
    print(f"nút {u}: con = {children[u]}, đánh dấu = {mark[u]}")

nút 0: con = {'b': 1, 'f': 5}, đánh dấu = -1
nút 1: con = {'a': 2}, đánh dấu = -1
nút 2: con = {'d': 3, 'g': 4}, đánh dấu = -1
nút 3: con = {}, đánh dấu = 0
nút 4: con = {}, đánh dấu = 1
nút 5: con = {'i': 6, 'e': 8}, đánh dấu = -1
nút 6: con = {'g': 7}, đánh dấu = -1
nút 7: con = {}, đánh dấu = 2
nút 8: con = {'d': 9}, đánh dấu = -1
nút 9: con = {}, đánh dấu = 3


## 2.2. Duyệt pre-order với thứ tự con cho trước

In [2]:
def preorder(children, mark, order_of=None):
    # order_of[u]: danh sách nút con theo thứ tự muốn duyệt (mặc định: thứ tự chèn)
    out = []
    stack = [0]
    while stack:
        u = stack.pop()
        if mark[u] != -1:
            out.append(mark[u])
        kids = (order_of or {}).get(u, list(children[u].values()))
        stack.extend(reversed(kids))          # đảo để con đầu tiên được lấy ra trước
    return out


print(preorder(children, mark))

[0, 1, 2, 3]


## 2.3. Đếm nghịch thế

Cách đơn giản `O(m²)`: đếm mọi cặp `a < b` có `seq[a] > seq[b]`.

Và một công cụ quan trọng: với hai danh sách **đã sắp** `A`, `B`, số cặp `(x ∈ A, y ∈ B)` có `y < x` bằng `Σ_{x ∈ A} bisect_left(B, x)` (số phần tử của `B` nhỏ hơn `x`).

In [3]:
from bisect import bisect_left, insort


def count_inversions(seq) -> int:
    return sum(1 for a in range(len(seq)) for b in range(a + 1, len(seq)) if seq[a] > seq[b])


def cross_count(A, B) -> int:
    # số cặp (x thuộc A, y thuộc B) với y < x ; A, B đã sắp tăng
    return sum(bisect_left(B, x) for x in A)


print(count_inversions([0, 3, 1, 2]))      # 2
print(cross_count([1, 2], [0, 3]))         # A đứng trước B: (1,0), (2,0) -> 2 nghịch thế
print(cross_count([0, 3], [1, 2]))         # B đứng trước A: (3,1), (3,2) -> 2

2
2
2


# 3. Cách 1 — Brute Force: thử mọi cách sắp con

In [4]:
from itertools import permutations, product


def min_inversions_bruteforce(words: List[str], target: List[int]) -> int:
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    branching = [u for u in range(len(children)) if len(children[u]) > 1]
    options = [list(permutations(children[u].values())) for u in branching]
    best = None
    for combo in product(*options):
        trav = preorder(children, mark, dict(zip(branching, combo)))
        cost = count_inversions([pos[i] for i in trav])
        best = cost if best is None else min(best, cost)
    return best


print(min_inversions_bruteforce(["bad", "bag", "fig", "fed"], [3, 1, 0, 2]))   # 2
print(min_inversions_bruteforce(["ab", "ac", "ba", "bc"], [0, 3, 1, 2]))       # 1

2
1


Số cách sắp là tích các `k!` trên mọi nút — bùng nổ cực nhanh.

---

# 4. Bước đột phá: số nghịch thế tách thành các phần độc lập

Xét một nút `u`. Trong `traversal`, các từ thuộc cây con của `u` nằm **liền một khối**: đầu tiên là từ của chính `u` (nếu có), sau đó lần lượt là khối của từng cây con.

Mọi cặp nghịch thế `(x, y)` thuộc đúng một trong các loại sau, xét tại **nút sâu nhất chứa cả hai** (gọi là `u`):

| Loại | Cặp | Phụ thuộc vào |
|---|---|---|
| (a) | `x` là từ của `u`, `y` trong một cây con của `u` | **không** phụ thuộc thứ tự con: `x` luôn đứng trước → nghịch thế khi `pos[y] < pos[x]` |
| (b) | `x`, `y` ở **hai cây con khác nhau** của `u` | **chỉ** phụ thuộc thứ tự giữa hai cây con đó tại `u` |

(Cặp nằm trong cùng một cây con của `u` thì nút sâu nhất chứa cả hai nằm sâu hơn.)

Hệ quả: tại mỗi nút, ta **tối ưu thứ tự con một cách độc lập**, chỉ cần biết **tập các giá trị `pos`** trong mỗi cây con con (không cần biết thứ tự bên trong chúng!).

Với hai cây con `A`, `B` (tập giá trị `pos`):

```text
cost[A][B] = số cặp nghịch thế nếu A đứng trước B = #{(x ∈ A, y ∈ B) : y < x}
cost[B][A] = |A|·|B| - cost[A][B]
```

Tổng chi phí loại (b) tại nút = `Σ cost[X][Y]` trên mọi cặp con mà `X` đứng trước `Y`.

# 5. Cách 2 — Thử mọi hoán vị con, nhưng riêng từng nút

Áp dụng mục 4: xử lý nút từ dưới lên, mỗi nút giữ **danh sách `pos` đã sắp** của cây con. Tại mỗi nút có `k >= 2` con, thử mọi hoán vị con (`k!`), lấy chi phí nhỏ nhất. Tổng hợp: cộng loại (a) và loại (b) của mọi nút.

Đã nhanh hơn brute force rất nhiều (các nút độc lập), nhưng `10! ≈ 3.6·10^6` hoán vị ở một nút vẫn quá chậm.

In [5]:
def min_inversions_perm(words: List[str], target: List[int]) -> int:
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    lists = [None] * len(children)
    total = 0
    for u in range(len(children) - 1, -1, -1):           # con trước cha
        L = [lists[v] for v in children[u].values()]
        k = len(L)
        if k >= 2:
            cost = [[cross_count(L[a], L[b]) for b in range(k)] for a in range(k)]
            total += min(sum(cost[order[i]][order[j]] for i in range(k) for j in range(i + 1, k))
                         for order in permutations(range(k)))
        merged = sorted(x for lst in L for x in lst)
        if mark[u] != -1:
            p = pos[mark[u]]
            total += bisect_left(merged, p)               # loại (a)
            insort(merged, p)
        lists[u] = merged
    return total


print(min_inversions_perm(["bad", "bag", "fig", "fed"], [3, 1, 0, 2]))
print(min_inversions_perm(["ab", "ac", "ba", "bc"], [0, 3, 1, 2]))

2
1


# 6. Bước đột phá 2: thứ tự tối ưu bằng bitmask DP

Bài toán con: có `k` khối, chi phí `cost[X][Y]` nếu `X` đứng trước `Y`. Tìm thứ tự có tổng chi phí nhỏ nhất.

Xây thứ tự **từ trái sang phải**. Trạng thái = **tập** các khối đã đặt (một bitmask `k` bit). Khi đặt thêm khối `c` vào **cuối**, mọi khối trong `mask` đều đứng trước `c`:

```text
dp[mask | (1 << c)] = min( dp[mask] + Σ_{a ∈ mask} cost[a][c] )
```

Đáp án là `dp[full]`. Điều quan trọng: chi phí thêm `c` **chỉ phụ thuộc tập** `mask`, không phụ thuộc thứ tự bên trong `mask` → DP đúng.

Để tính nhanh `add[mask][c] = Σ_{a ∈ mask} cost[a][c]`, dựng tăng dần theo bit thấp nhất: `add[mask][c] = add[mask bỏ bit thấp nhất][c] + cost[bit thấp nhất][c]`.

Số trạng thái `2^10 = 1024`, mỗi trạng thái `k` lựa chọn → rất nhanh.

In [6]:
def best_order_cost(cost: List[List[int]]) -> int:
    k = len(cost)
    full = 1 << k
    add = [[0] * k for _ in range(full)]
    for mask in range(1, full):
        low = (mask & -mask).bit_length() - 1             # chỉ số bit thấp nhất
        prev = mask & (mask - 1)
        for c in range(k):
            add[mask][c] = add[prev][c] + cost[low][c]
    INF = float("inf")
    dp = [INF] * full
    dp[0] = 0
    for mask in range(full):
        if dp[mask] == INF:
            continue
        for c in range(k):
            if not mask >> c & 1:
                nm = mask | (1 << c)
                if dp[mask] + add[mask][c] < dp[nm]:
                    dp[nm] = dp[mask] + add[mask][c]
    return dp[full - 1]


cost = [[0, 5, 1], [2, 0, 4], [3, 1, 0]]
brute = min(sum(cost[o[i]][o[j]] for i in range(3) for j in range(i + 1, 3)) for o in permutations(range(3)))
print("bitmask DP:", best_order_cost(cost), "| thử mọi hoán vị:", brute)

bitmask DP: 6 | thử mọi hoán vị: 6


Mẹo `mask & -mask` lấy ra bit 1 thấp nhất; `mask & (mask - 1)` xóa bit đó.

# 7. Cách 3 — Trie + danh sách đã sắp + bitmask DP

In [7]:
def min_inversions_fast(words: List[str], target: List[int], verbose: bool = False) -> int:
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    lists = [None] * len(children)
    total = 0
    for u in range(len(children) - 1, -1, -1):
        items = list(children[u].items())
        L = [lists[v] for _, v in items]
        k = len(L)
        if k >= 2:
            cost = [[0] * k for _ in range(k)]
            for a in range(k):
                for b in range(a + 1, k):
                    c = cross_count(L[a], L[b])
                    cost[a][b] = c
                    cost[b][a] = len(L[a]) * len(L[b]) - c
            part = best_order_cost(cost)
            total += part
            if verbose:
                print(f"nút {u}: các con {[ch for ch, _ in items]} với pos {L}, cost = {cost} -> +{part}")
        merged = sorted(x for lst in L for x in lst)
        if mark[u] != -1:
            p = pos[mark[u]]
            own = bisect_left(merged, p)
            total += own
            if verbose and merged:
                print(f"nút {u}: từ '{words[mark[u]]}' (pos {p}) đứng trước {merged} -> +{own}")
            insort(merged, p)
        lists[u] = merged
        for v in children[u].values():
            lists[v] = None                                # giải phóng bộ nhớ
    return total


print(min_inversions_fast(["bad", "bag", "fig", "fed"], [3, 1, 0, 2], verbose=True))

nút 5: các con ['i', 'e'] với pos [[3], [0]], cost = [[0, 1], [0, 0]] -> +0
nút 2: các con ['d', 'g'] với pos [[2], [1]], cost = [[0, 1], [0, 0]] -> +0
nút 0: các con ['b', 'f'] với pos [[1, 2], [0, 3]], cost = [[0, 2], [2, 0]] -> +2
2


## Mô phỏng ví dụ 1

`pos`: `"bad" → 2`, `"bag" → 1`, `"fig" → 3`, `"fed" → 0`.

| Nút | Các con và tập `pos` | cost (đứng trước → nghịch thế) | tốt nhất |
|---|---|---|---:|
| `"ba"` | `d: [2]`, `g: [1]` | `d` trước `g`: 1; `g` trước `d`: 0 | 0 |
| `"f"` | `i: [3]`, `e: [0]` | `i` trước `e`: 1; `e` trước `i`: 0 | 0 |
| gốc | `b: [1, 2]`, `f: [0, 3]` | `b` trước `f`: 2 (`1>0`, `2>0`); `f` trước `b`: 2 (`3>1`, `3>2`) | 2 |

Tổng `2` ✔. (Không nút nào vừa đánh dấu vừa có con, nên không có loại (a).)

# 8. Phiên bản đúng format LeetCode

In [8]:
from bisect import bisect_left, insort
from typing import List


class Solution:
    def minInversions(self, words: List[str], target: List[int]) -> int:
        pos = [0] * len(target)
        for k, v in enumerate(target):
            pos[v] = k
        children, mark = [{}], [-1]
        for i, w in enumerate(words):
            u = 0
            for c in w:
                if c not in children[u]:
                    children[u][c] = len(children)
                    children.append({})
                    mark.append(-1)
                u = children[u][c]
            mark[u] = i

        INF = float("inf")
        lists = [None] * len(children)
        total = 0
        for u in range(len(children) - 1, -1, -1):
            L = [lists[v] for v in children[u].values()]
            k = len(L)
            if k >= 2:
                cost = [[0] * k for _ in range(k)]
                for a in range(k):
                    for b in range(a + 1, k):
                        c = 0
                        B = L[b]
                        for x in L[a]:
                            c += bisect_left(B, x)
                        cost[a][b] = c
                        cost[b][a] = len(L[a]) * len(B) - c
                full = 1 << k
                add = [[0] * k for _ in range(full)]
                for mask in range(1, full):
                    low = (mask & -mask).bit_length() - 1
                    prev, row = add[mask & (mask - 1)], cost[low]
                    add[mask] = [prev[c] + row[c] for c in range(k)]
                dp = [INF] * full
                dp[0] = 0
                for mask in range(full):
                    d = dp[mask]
                    if d == INF:
                        continue
                    ad = add[mask]
                    for c in range(k):
                        if not mask >> c & 1:
                            nm = mask | (1 << c)
                            if d + ad[c] < dp[nm]:
                                dp[nm] = d + ad[c]
                total += dp[full - 1]
            merged = sorted(x for lst in L for x in lst)
            if mark[u] != -1:
                p = pos[mark[u]]
                total += bisect_left(merged, p)
                insort(merged, p)
            lists[u] = merged
            for v in children[u].values():
                lists[v] = None
        return total

In [9]:
sol = Solution()
tests = [
    (["bad", "bag", "fig", "fed"], [3, 1, 0, 2], 2, "ví dụ 1"),
    (["ab", "ac", "ba", "bc"], [0, 3, 1, 2], 1, "ví dụ 2"),
    (["a"], [0], 0, "một từ"),
    (["a", "ab", "abc"], [2, 1, 0], 3, "chuỗi lồng nhau: thứ tự bắt buộc"),
    (["a", "b", "c"], [2, 0, 1], 0, "ba con của gốc: sắp được hoàn hảo"),
]
for w, t, expected, name in tests:
    result = sol.minInversions(w, t)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:36} -> {result}")

PASS | ví dụ 1                              -> 2
PASS | ví dụ 2                              -> 1
PASS | một từ                               -> 0
PASS | chuỗi lồng nhau: thứ tự bắt buộc     -> 3
PASS | ba con của gốc: sắp được hoàn hảo    -> 0


Test `["a", "ab", "abc"]`: mỗi từ là tiền tố của từ sau nên thứ tự duyệt **bắt buộc** là `a, ab, abc` (từ ở nút được ghi trước cây con), dãy `pos = [2, 1, 0]` có 3 nghịch thế — đó là loại (a), không thể tránh.

# 9. Vì sao đúng?

1. **Phân loại cặp:** mỗi cặp từ có duy nhất một nút sâu nhất chứa cả hai; cặp đó thuộc loại (a) (một từ chính là nút ấy) hoặc loại (b) (hai cây con khác nhau).
2. **Loại (a)** cố định: từ ở nút luôn được ghi trước toàn bộ cây con.
3. **Loại (b)** tại nút `u` chỉ phụ thuộc thứ tự con tại `u`, và chỉ phụ thuộc **tập** `pos` của mỗi cây con (mọi phần tử khối trước đứng trước mọi phần tử khối sau).
4. Vì các nút độc lập, tổng nhỏ nhất = tổng (a) + Σ (min loại (b) tại từng nút).
5. **Bitmask DP** đúng: chi phí thêm khối `c` vào cuối chỉ phụ thuộc tập đã đặt.

# 10. Độ phức tạp

Gọi `n` = số từ, `ℓ` = độ dài từ tối đa (20), `k` = số con tối đa (10).

| Cách | Ý tưởng | Thời gian |
|---|---|---:|
| 1. Brute Force | mọi cách sắp ở mọi nút | `Π (k_u)!` × `O(n²)` |
| 2. Hoán vị từng nút | nút độc lập, `k!` mỗi nút | `O(n ℓ log n + Σ k_u! · k_u²)` |
| 3. Bitmask DP | `2^k · k` mỗi nút | `O(n ℓ log n + Σ 2^{k_u} · k_u)` |

Phần `n ℓ log n`: mỗi từ có mặt trong danh sách của tối đa `ℓ + 1` nút tổ tiên. Số nút có `k` con không quá `n / (k - 1)`, nên phần bitmask cũng nhỏ.

# 11. Đối chiếu ba cách trên dữ liệu ngẫu nhiên

In [10]:
import random
import time

random.seed(4077)
for _ in range(1000):
    n = random.randint(1, 6)
    ws = set()
    while len(ws) < n:
        ws.add("".join(random.choice("abc") for _ in range(random.randint(1, 3))))
    ws = list(ws)
    t = list(range(n))
    random.shuffle(t)
    expected = min_inversions_bruteforce(ws, t)
    assert min_inversions_perm(ws, t) == expected, (ws, t)
    assert min_inversions_fast(ws, t) == expected, (ws, t)
    assert sol.minInversions(ws, t) == expected, (ws, t)
print("1000 bộ test ngẫu nhiên: 4 hàm cho cùng kết quả!")

ws = set()
while len(ws) < 10**4:
    ws.add("".join(random.choice("abcdefghij") for _ in range(random.randint(1, 20))))
ws = list(ws)
t = list(range(len(ws)))
random.shuffle(t)
t0 = time.perf_counter()
print("n = 10^4:", sol.minInversions(ws, t), f"({time.perf_counter() - t0:.2f}s)")

1000 bộ test ngẫu nhiên: 4 hàm cho cùng kết quả!


n = 10^4: 24027035 (0.42s)


# 12. Những lỗi dễ mắc

## Lỗi 1 — Sắp con theo một tiêu chí đơn giản (ví dụ theo `min pos` hoặc trung bình)

Nghe hợp lý ("khối có giá trị nhỏ đi trước") nhưng không luôn tối ưu. Tìm phản ví dụ tự động:

In [11]:
def wrong_sort_by_min(words, target):
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    lists = [None] * len(children)
    total = 0
    for u in range(len(children) - 1, -1, -1):
        L = sorted((lists[v] for v in children[u].values()), key=lambda l: l[0])   # SAI: sắp theo min
        for i in range(len(L)):
            for j in range(i + 1, len(L)):
                total += cross_count(L[i], L[j])
        merged = sorted(x for lst in L for x in lst)
        if mark[u] != -1:
            p = pos[mark[u]]
            total += bisect_left(merged, p)
            insort(merged, p)
        lists[u] = merged
    return total


random.seed(1)
for _ in range(3000):
    n = random.randint(2, 7)
    ws = set()
    while len(ws) < n:
        ws.add("".join(random.choice("abc") for _ in range(random.randint(1, 3))))
    ws = list(ws); t = list(range(n)); random.shuffle(t)
    if wrong_sort_by_min(ws, t) != sol.minInversions(ws, t):
        print("Phản ví dụ:", ws, t, "| sai:", wrong_sort_by_min(ws, t), "| đúng:", sol.minInversions(ws, t))
        break

Phản ví dụ: ['bac', 'aa', 'cab', 'cbc', 'cba', 'c'] [4, 1, 3, 5, 2, 0] | sai: 5 | đúng: 3


## Lỗi 2 — Quên loại (a): từ ở nút đứng trước cả cây con

Nếu chỉ tính chi phí giữa các cây con, test `["a", "ab", "abc"]` ra `0` thay vì `3`.

In [12]:
def wrong_no_own_word(words, target):
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    lists = [None] * len(children)
    total = 0
    for u in range(len(children) - 1, -1, -1):
        L = [lists[v] for v in children[u].values()]
        if len(L) >= 2:
            k = len(L)
            cost = [[cross_count(L[a], L[b]) if a != b else 0 for b in range(k)] for a in range(k)]
            total += best_order_cost(cost)
        merged = sorted(x for lst in L for x in lst)
        if mark[u] != -1:
            insort(merged, pos[mark[u]])          # SAI: không cộng số nghịch thế với cây con
        lists[u] = merged
    return total


print("Sai :", wrong_no_own_word(["a", "ab", "abc"], [2, 1, 0]))
print("Đúng:", sol.minInversions(["a", "ab", "abc"], [2, 1, 0]))

Sai : 0
Đúng: 3


## Lỗi 3 — Dùng chỉ số từ thay vì `pos`

Nghịch thế được định nghĩa trên `pos[traversal[a]]` (vị trí trong `target`), không phải trên chính chỉ số từ. Nhầm hai thứ này cho kết quả sai ở ví dụ 1.

## Lỗi 4 — Thử `k!` hoán vị ở nút có 10 con

`10! = 3.6·10^6` hoán vị × `k²` chi phí mỗi hoán vị — quá chậm. Bitmask DP chỉ cần `2^10 · 10 ≈ 10^4`.

## Lỗi 5 — Tính `cost[A][B]` bằng hai vòng lặp lồng nhau trên từng phần tử

`O(|A| · |B|)` có thể rất lớn ở gốc. Dùng danh sách đã sắp + `bisect` (hoặc hai con trỏ).

# 13. Mẹo nhận diện

1. Đếm cặp / nghịch thế trên một cây duyệt theo thứ tự tùy chọn → xét **nút sâu nhất chứa cả hai** để tách tổng thành các phần độc lập.
2. "Chọn thứ tự của `k` khối, chi phí theo **cặp** (ai đứng trước ai)" với `k` nhỏ (≤ 20) → **bitmask DP**: `dp[mask] + Σ cost[a ∈ mask][c]`.
3. Đếm cặp `(x ∈ A, y ∈ B)` với `y < x` → sắp hai danh sách, `bisect` / hai con trỏ / merge sort.
4. Bảng chữ cái nhỏ (`a..j`) là gợi ý: số con mỗi nút ≤ 10 → `2^10` chấp nhận được.

Các bài liên quan: *Count of Smaller Numbers After Self* (LeetCode 315), *Global and Local Inversions* (775), *Find the Shortest Superstring* (943 — bitmask DP trên thứ tự).

# 14. Biến thể

## Biến thể 1 — Tối đa số nghịch thế

Đổi `min` thành `max` trong bitmask DP (loại (a) vẫn cố định).

In [13]:
def max_inversions(words, target):
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    lists = [None] * len(children)
    total = 0
    for u in range(len(children) - 1, -1, -1):
        L = [lists[v] for v in children[u].values()]
        k = len(L)
        if k >= 2:
            cost = [[-cross_count(L[a], L[b]) if a != b else 0 for b in range(k)] for a in range(k)]
            total += -best_order_cost(cost)                 # max = -min(-cost)
        merged = sorted(x for lst in L for x in lst)
        if mark[u] != -1:
            total += bisect_left(merged, pos[mark[u]])         # loại (a) không đổi
            insort(merged, pos[mark[u]])
        lists[u] = merged
    return total


print(max_inversions(["bad", "bag", "fig", "fed"], [3, 1, 0, 2]))

4


## Biến thể 2 — Bài toán sắp xếp khối với chi phí theo cặp, đứng độc lập

`best_order_cost` dùng được cho mọi bài "sắp `k` vật, chi phí khi vật `X` đứng trước vật `Y`", ví dụ ghép chuỗi sao cho số cặp ký tự `"ba"` ít nhất.

# 15. Template tư duy

```python
for u in nodes_bottom_up:
    L = [sorted_pos_list(child) for child in children(u)]
    cost[a][b] = cross_count(L[a], L[b]); cost[b][a] = |a|·|b| - cost[a][b]
    total += bitmask_dp_min_order(cost)                # loại (b)
    merged = merge(L)
    if u có từ: total += bisect_left(merged, pos[u]); chèn pos[u]   # loại (a)
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Ví dụ 2 (`["ab","ac","ba","bc"]`, `target = [0,3,1,2]`): tính `pos`, các tập `pos` của từng cây con và chi phí ở mỗi nút. Kiểm tra bằng cell dưới.

In [14]:
# Kiểm tra Bài 1
print(min_inversions_fast(["ab", "ac", "ba", "bc"], [0, 3, 1, 2], verbose=True))

nút 4: các con ['a', 'c'] với pos [[3], [1]], cost = [[0, 1], [0, 0]] -> +0
nút 1: các con ['b', 'c'] với pos [[0], [2]], cost = [[0, 0], [1, 0]] -> +0
nút 0: các con ['a', 'b'] với pos [[0, 2], [1, 3]], cost = [[0, 1], [3, 0]] -> +1
1


## Bài 2 — Dự đoán output

```python
print(bisect_left([1, 3, 5], 4), bisect_left([1, 3, 5], 1), bisect_left([1, 3, 5], 9))
m = 0b10110
print(m & -m, m & (m - 1), (m & -m).bit_length() - 1)
```

In [15]:
print(bisect_left([1, 3, 5], 4), bisect_left([1, 3, 5], 1), bisect_left([1, 3, 5], 9))
m = 0b10110
print(m & -m, m & (m - 1), (m & -m).bit_length() - 1)

2 0 3
2 20 1


## Bài 3 — Hoàn thiện `cross_count` bằng hai con trỏ (không dùng `bisect`)

In [16]:
def exercise_3_cross_count(A, B) -> int:
    # TODO: A, B đã sắp; đếm cặp (x ∈ A, y ∈ B) với y < x bằng một con trỏ trên B
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_cross_count([1, 2], [0, 3]) == 2

## Bài 4 — Đếm nghịch thế `O(n log n)` bằng merge sort

```python
count_inv_fast([0, 3, 1, 2])   -> 2
count_inv_fast([5, 4, 3, 2, 1]) -> 10
```

In [17]:
def exercise_4_count_inv_fast(seq) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_count_inv_fast([0, 3, 1, 2]) == 2
# assert exercise_4_count_inv_fast([5, 4, 3, 2, 1]) == 10

## Bài 5 — Sắp chuỗi nhị phân để ít cặp `(1, 0)` nhất

Cho các chuỗi nhị phân; nối chúng theo thứ tự tùy chọn. Đếm số cặp vị trí `a < b` có ký tự `1` ở `a`, `0` ở `b`. Tìm số nhỏ nhất (`k <= 10` chuỗi), dùng `best_order_cost`.

```python
min_one_zero_pairs(["10", "0", "1"])   -> 1     # "0" + "10" + "1": chỉ cặp bên trong "10"
```

In [18]:
def exercise_5_min_one_zero_pairs(strs: List[str]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_min_one_zero_pairs(["10", "0", "1"]) == 1

## Bài 6 — Mini challenge: trả về `traversal` tối ưu

Truy vết bitmask DP tại mỗi nút để lấy thứ tự con, rồi duyệt pre-order. Kiểm tra: số nghịch thế của dãy `pos` thu được bằng đáp án.

In [19]:
def exercise_6_best_traversal(words: List[str], target: List[int]) -> List[int]:
    # TODO
    raise NotImplementedError


# trav = exercise_6_best_traversal(["bad", "bag", "fig", "fed"], [3, 1, 0, 2])
# pos = {v: k for k, v in enumerate([3, 1, 0, 2])}
# assert count_inversions([pos[i] for i in trav]) == 2

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: duyệt `x` trong `A` tăng dần; con trỏ `j` trên `B` chỉ tiến.
- Mức 2: `while j < len(B) and B[j] < x: j += 1`.
- Mức 3: cộng `j` cho mỗi `x`.

**Bài 4**

- Mức 1: chia đôi, đếm hai nửa đệ quy, rồi đếm cặp chéo khi trộn.
- Mức 2: khi lấy phần tử từ nửa phải (nhỏ hơn), mọi phần tử còn lại của nửa trái đều tạo nghịch thế.
- Mức 3: cộng `len(left) - i` mỗi lần lấy từ phải.

**Bài 5**

- Mức 1: chi phí bên trong mỗi chuỗi cố định.
- Mức 2: `cost[X][Y]` = (số `1` trong `X`) × (số `0` trong `Y`).
- Mức 3: đáp án = Σ chi phí trong + `best_order_cost(cost)`.

**Bài 6**

- Mức 1: lưu `dp` và vết `parent[mask] = c` cuối cùng được thêm.
- Mức 2: từ `full`, lần ngược để có thứ tự (đảo lại).
- Mức 3: ghép thành `order_of[u]` rồi gọi `preorder`.

# 17. Đáp án tham khảo

## Đáp án Bài 1

`pos`: `"ab" → 0`, `"ac" → 2`, `"ba" → 3`, `"bc" → 1`.

| Nút | Con và tập `pos` | cost | tốt nhất |
|---|---|---|---:|
| `"a"` | `b: [0]`, `c: [2]` | `b` trước `c`: 0; ngược lại: 1 | 0 |
| `"b"` | `a: [3]`, `c: [1]` | `a` trước `c`: 1; ngược lại: 0 | 0 |
| gốc | `a: [0, 2]`, `b: [1, 3]` | `a` trước `b`: 1 (`2 > 1`); ngược lại: 3 | 1 |

Tổng `1` ✔.

## Đáp án Bài 2

```text
2 0 3
2 20 1
```

In [20]:
# Đáp án Bài 3
def answer_3_cross_count(A, B) -> int:
    j = total = 0
    for x in A:
        while j < len(B) and B[j] < x:
            j += 1
        total += j
    return total


assert answer_3_cross_count([1, 2], [0, 3]) == 2
random.seed(3)
for _ in range(500):
    A = sorted(random.sample(range(30), random.randint(0, 8)))
    B = sorted(random.sample(range(30), random.randint(0, 8)))
    assert answer_3_cross_count(A, B) == cross_count(A, B)
print("Bài 3: PASS")

Bài 3: PASS


In [21]:
# Đáp án Bài 4
def answer_4_count_inv_fast(seq) -> int:
    def sort_count(a):
        if len(a) <= 1:
            return a, 0
        mid = len(a) // 2
        left, x = sort_count(a[:mid])
        right, y = sort_count(a[mid:])
        merged, i, j, cnt = [], 0, 0, x + y
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                merged.append(right[j]); j += 1
                cnt += len(left) - i
        merged += left[i:] + right[j:]
        return merged, cnt
    return sort_count(list(seq))[1]


assert answer_4_count_inv_fast([0, 3, 1, 2]) == 2
assert answer_4_count_inv_fast([5, 4, 3, 2, 1]) == 10
print("Bài 4: PASS")

Bài 4: PASS


In [22]:
# Đáp án Bài 5
def answer_5_min_one_zero_pairs(strs: List[str]) -> int:
    inside = sum(count_inversions([int(ch) for ch in s]) for s in strs)
    ones = [s.count("1") for s in strs]
    zeros = [s.count("0") for s in strs]
    k = len(strs)
    cost = [[ones[a] * zeros[b] if a != b else 0 for b in range(k)] for a in range(k)]
    return inside + best_order_cost(cost)


def brute_one_zero(strs):
    return min(count_inversions([int(ch) for ch in "".join(strs[i] for i in o)]) for o in permutations(range(len(strs))))


assert answer_5_min_one_zero_pairs(["10", "0", "1"]) == 1
random.seed(5)
for _ in range(300):
    strs = ["".join(random.choice("01") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 5))]
    assert answer_5_min_one_zero_pairs(strs) == brute_one_zero(strs)
print("Bài 5: PASS (khớp brute force trên 300 test)")

Bài 5: PASS (khớp brute force trên 300 test)


In [23]:
# Đáp án Bài 6
def answer_6_best_traversal(words: List[str], target: List[int]) -> List[int]:
    pos = [0] * len(target)
    for k, v in enumerate(target):
        pos[v] = k
    children, mark = build_trie(words)
    lists = [None] * len(children)
    order_of = {}
    for u in range(len(children) - 1, -1, -1):
        kids = list(children[u].values())
        L = [lists[v] for v in kids]
        k = len(L)
        if k >= 2:
            cost = [[cross_count(L[a], L[b]) if a != b else 0 for b in range(k)] for a in range(k)]
            full = 1 << k
            INF = float("inf")
            dp, par = [INF] * full, [-1] * full
            dp[0] = 0
            for mask in range(full):
                if dp[mask] == INF:
                    continue
                for c in range(k):
                    if not mask >> c & 1:
                        val = dp[mask] + sum(cost[a][c] for a in range(k) if mask >> a & 1)
                        if val < dp[mask | 1 << c]:
                            dp[mask | 1 << c], par[mask | 1 << c] = val, c
            seq, mask = [], full - 1
            while mask:
                c = par[mask]
                seq.append(kids[c])
                mask ^= 1 << c
            order_of[u] = seq[::-1]
        merged = sorted(x for lst in L for x in lst)
        if mark[u] != -1:
            insort(merged, pos[mark[u]])
        lists[u] = merged
    return preorder(children, mark, order_of)


trav = answer_6_best_traversal(["bad", "bag", "fig", "fed"], [3, 1, 0, 2])
pos_map = {v: k for k, v in enumerate([3, 1, 0, 2])}
print("traversal:", trav)
assert count_inversions([pos_map[i] for i in trav]) == 2
random.seed(6)
for _ in range(300):
    n = random.randint(1, 7)
    ws = set()
    while len(ws) < n:
        ws.add("".join(random.choice("abc") for _ in range(random.randint(1, 3))))
    ws = list(ws); t = list(range(n)); random.shuffle(t)
    pm = {v: k for k, v in enumerate(t)}
    tr = answer_6_best_traversal(ws, t)
    assert sorted(tr) == list(range(n)) and count_inversions([pm[i] for i in tr]) == sol.minInversions(ws, t)
print("Bài 6: PASS")

traversal: [1, 0, 3, 2]
Bài 6: PASS


# 18. Tổng kết

**Minimize Inversions in a Prefix Tree**:

1. Trie + duyệt pre-order; mỗi cây con là một **khối liền**.
2. Mọi cặp nghịch thế quy về nút sâu nhất chứa nó → các nút **độc lập**.
3. Loại (a) (từ ở nút vs cây con) cố định: `bisect_left`.
4. Loại (b) (giữa các cây con anh em): `cost` theo cặp từ danh sách đã sắp, thứ tự tối ưu bằng **bitmask DP** (`k <= 10`).

## Độ phức tạp

```text
Time:  O(n · ℓ · log n + Σ 2^k · k)
Space: O(n · ℓ)
```

# 19. Checklist trước khi submit

- [ ] Nghịch thế tính trên `pos` (vị trí trong `target`), không phải chỉ số từ?
- [ ] Cộng loại (a): từ ở nút đứng trước toàn bộ cây con?
- [ ] `cost[b][a] = |A|·|B| - cost[a][b]`?
- [ ] Bitmask DP thay vì thử `k!` hoán vị?
- [ ] Xử lý nút từ dưới lên (số nút giảm dần)?
- [ ] Đã test: một từ, chuỗi tiền tố lồng nhau, nhiều con ở gốc?